# PySpark drills 11: structured streaming

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-drills-11-structured-streaming.ipynb)

Practice for [PySpark 11: structured streaming](pyspark-11-structured-streaming.ipynb). Each data cell lands JSON files under `WORK_DIR`; your answer streams them with `availableNow`.

Write your answer in the empty cell under each exercise, run it, and compare your output with the expected output.
The reference answer is hidden under the exercise; open it when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

import json


def land_json(directory, name, rows, mtime):
    """Write one JSON-lines file with a fixed modification time, so files are processed in order."""
    os.makedirs(directory, exist_ok=True)
    path = os.path.join(directory, name)
    with open(path, "w") as handle:
        handle.write("\n".join(json.dumps(row) for row in rows))
    os.utime(path, (mtime, mtime))


def run_until_done(query, timeout=90):
    try:
        if not query.awaitTermination(timeout):
            raise TimeoutError("stream did not finish")
    finally:
        query.stop()


customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

## Exercise 1: Deduplicate a file stream

**Difficulty:** Medium · about 15 min

Stream the files in `landing` (schema `event_id string, event_ts timestamp`) one file per micro-batch, drop repeated
`event_id`s within a 30-minute watermark, and write to a memory table `drill_events`. Return its distinct `event_id`s
ordered.

**Expected output**

| event_id |
| --- |
| e1 |
| e2 |
| e3 |

In [ ]:
# Sample data for this exercise.
landing = os.path.join(WORK_DIR, "drill_stream_dedup")
land_json(landing, "a.json", [{"event_id": "e1", "event_ts": "2026-01-06 09:00:00"}, {"event_id": "e2", "event_ts": "2026-01-06 09:01:00"}], 1_000)
land_json(landing, "b.json", [{"event_id": "e2", "event_ts": "2026-01-06 09:01:00"}, {"event_id": "e3", "event_ts": "2026-01-06 09:03:00"}], 2_000)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
query = (
    spark.readStream.schema("event_id string, event_ts timestamp").option("maxFilesPerTrigger", 1).json(landing)
    .withWatermark("event_ts", "30 minutes").dropDuplicatesWithinWatermark(["event_id"])
    .writeStream.format("memory").queryName("drill_events").outputMode("append")
    .option("checkpointLocation", os.path.join(WORK_DIR, "drill_dedup_checkpoint"))
    .trigger(availableNow=True).start()
)
run_until_done(query)
result = spark.table("drill_events").select("event_id").orderBy("event_id")
result.show()

## Exercise 2: Five-minute windows

**Difficulty:** Medium · about 15 min

Stream `clicks` (schema `ts timestamp, user_id int`) into five-minute tumbling windows in `complete` mode, counting
clicks. Return `window_start` (formatted `HH:mm`) and `clicks`, ordered by `window_start`.

**Expected output**

| window_start | clicks |
| --- | --- |
| 10:00 | 2 |
| 10:05 | 1 |
| 10:10 | 1 |

In [ ]:
# Sample data for this exercise.
clicks = os.path.join(WORK_DIR, "drill_clicks")
land_json(clicks, "a.json", [{"ts": "2026-01-06 10:01:00", "user_id": 1}, {"ts": "2026-01-06 10:04:00", "user_id": 2},
                             {"ts": "2026-01-06 10:06:00", "user_id": 1}, {"ts": "2026-01-06 10:14:00", "user_id": 3}], 1_000)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
counts = (
    spark.readStream.schema("ts timestamp, user_id int").json(clicks)
    .groupBy(F.window("ts", "5 minutes")).agg(F.count("*").alias("clicks"))
)
query = (
    counts.writeStream.format("memory").queryName("drill_windows").outputMode("complete")
    .option("checkpointLocation", os.path.join(WORK_DIR, "drill_windows_checkpoint"))
    .trigger(availableNow=True).start()
)
run_until_done(query)
result = spark.table("drill_windows").select(
    F.date_format("window.start", "HH:mm").alias("window_start"), "clicks"
).orderBy("window_start")
result.show()

## Exercise 3: Enrich a stream with a dimension

**Difficulty:** Medium · about 10 min

Stream `events_dir` (schema `event_id string, user_id int`) and left-join each event to `customers` on
`user_id = customer_id` to add `region`. Write to a memory table and return `event_id` and `region` ordered by
`event_id`.

**Expected output**

| event_id | region |
| --- | --- |
| x1 | North |
| x2 | West |
| x3 | null |

In [ ]:
# Sample data for this exercise.
events_dir = os.path.join(WORK_DIR, "drill_stream_events")
land_json(events_dir, "a.json", [{"event_id": "x1", "user_id": 2}, {"event_id": "x2", "user_id": 5}, {"event_id": "x3", "user_id": 42}], 1_000)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
regions = customers.select(F.col("customer_id").alias("user_id"), "region")
query = (
    spark.readStream.schema("event_id string, user_id int").json(events_dir)
    .join(regions, "user_id", "left")
    .writeStream.format("memory").queryName("drill_enriched").outputMode("append")
    .option("checkpointLocation", os.path.join(WORK_DIR, "drill_enriched_checkpoint"))
    .trigger(availableNow=True).start()
)
run_until_done(query)
result = spark.table("drill_enriched").select("event_id", "region").orderBy("event_id")
result.show()

## Exercise 4: Session windows

**Difficulty:** Hard · about 20 min

Stream `activity` (schema `user_id int, ts timestamp`) one file per micro-batch into session windows with a 10-minute
gap and a 10-minute watermark, in `append` mode. Return `user_id`, `start` and `end` (formatted `HH:mm`) and
`events` for the sessions emitted, ordered by `user_id` then `start`.

**Expected output**

| user_id | start | end | events |
| --- | --- | --- | --- |
| 1 | 10:00 | 10:18 | 2 |
| 2 | 10:30 | 10:40 | 1 |

In [ ]:
# Sample data for this exercise.
activity = os.path.join(WORK_DIR, "drill_activity")
land_json(activity, "a.json", [{"user_id": 1, "ts": "2026-01-06 10:00:00"}, {"user_id": 1, "ts": "2026-01-06 10:08:00"},
                               {"user_id": 2, "ts": "2026-01-06 10:30:00"}], 1_000)
land_json(activity, "b.json", [{"user_id": 3, "ts": "2026-01-06 12:00:00"}], 2_000)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
sessions = (
    spark.readStream.schema("user_id int, ts timestamp").option("maxFilesPerTrigger", 1).json(activity)
    .withWatermark("ts", "10 minutes")
    .groupBy("user_id", F.session_window("ts", "10 minutes")).agg(F.count("*").alias("events"))
)
query = (
    sessions.writeStream.format("memory").queryName("drill_sessions").outputMode("append")
    .option("checkpointLocation", os.path.join(WORK_DIR, "drill_sessions_checkpoint"))
    .trigger(availableNow=True).start()
)
run_until_done(query)
result = spark.table("drill_sessions").select(
    "user_id",
    F.date_format("session_window.start", "HH:mm").alias("start"),
    F.date_format("session_window.end", "HH:mm").alias("end"),
    "events",
).orderBy("user_id", "start")
result.show()

**Next:** [PySpark 12: internals and monitoring](pyspark-12-internals-and-monitoring.ipynb)